In [13]:
!pip install ultralytics

In [6]:
import torch
print(f"Is GPU available? {torch.cuda.is_available()}")
# This should print: Is GPU available? True
# if False, change runtime type (top right downwards arrow) to T4 GPU, restart session

Is GPU available? True


In [7]:
import zipfile
import os

# must have archive.zip (the dataset) in /content (drag it into 'files' on left menu)
# makes datasets folder and puts unzipped archive into datasets
os.makedirs('/content/datasets', exist_ok=True)
with zipfile.ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/datasets/archive')

In [9]:
import random
import shutil
import cv2
import os
import numpy as np
from pathlib import Path

def apply_digital_tape(image, bbox, color=(50, 50, 50), thickness_ratio=0.2):
    """
    Applies a horizontal tape strip across center of the bounding box.
    """
    h, w, _ = image.shape
    x_center, y_center, b_w, b_h = bbox

    # converts normalized YOLO coordinates to pixel coordinates
    x1 = int((x_center - b_w / 2) * w)
    y1 = int((y_center - b_h / 2) * h)
    x2 = int((x_center + b_w / 2) * w)
    y2 = int((y_center + b_h / 2) * h)

    # define tape dimensions (horizontal strip across middle)
    tape_h = int((y2 - y1) * thickness_ratio)
    tape_y1 = y_center * h - (tape_h / 2)
    tape_y2 = y_center * h + (tape_h / 2)

    # draw tape
    cv2.rectangle(image, (x1, int(tape_y1)), (x2, int(tape_y2)), color, -1)
    return image

def apply_adversarial_patch(image, bbox):
  h, w, _ = image.shape
  x_center, y_center, b_w, b_h = bbox

  # convert to pixel coordinates
  x1 = int((x_center - b_w / 4) * w)
  y1 = int((y_center - b_h / 4) * h)
  x2 = int((x_center + b_w / 4) * w)
  y2 = int((y_center + b_h / 4) * h)

  # gen adversarial noise
  noise_patch = np.random.randint(0, 255, (y2-y1, x2-x1, 3), dtype=np.uint8)

  # overlay the noise onto sign
  image[y1:y2, x1:x2] = noise_patch
  return image

def apply_heavy_occlusion(image, bbox):
  h, w, _ = image.shape
  x_center, y_center, b_w, b_h = bbox

  x1 = int((x_center - b_w * 0.4) * w)
  y1 = int((y_center - b_h * 0.4) * h)
  x2 = int((x_center + b_w * 0.4) * w)
  y2 = int((y_center + b_h * 0.4) * h)

  cv2.rectangle(image, (x1, y1), (x2, y2), (128, 128, 128), -1)
  return image

val_images_path = '/content/datasets/archive/valid/images'
val_labels_path = '/content/datasets/archive/valid/labels'
output_path = '/content/datasets/archive/attacked_val/images'
os.makedirs(output_path, exist_ok=True)

def create_robust_train_set(base_path, split_ratio=0.5, attack_type="tape"):
    train_images_src = os.path.join(base_path, 'train/images')
    train_labels_src = os.path.join(base_path, 'train/labels')

    # New folder for the 50/50 mix
    robust_root = os.path.join(base_path, 'train_robust')
    os.makedirs(os.path.join(robust_root, 'images'), exist_ok=True)
    os.makedirs(os.path.join(robust_root, 'labels'), exist_ok=True)

    all_images = [f for f in os.listdir(train_images_src) if f.endswith(('.jpg', '.png'))]
    random.shuffle(all_images)

    split_idx = int(len(all_images) * split_ratio)
    attack_list = all_images[:split_idx]

    print(f"Creating robust set: {len(attack_list)} attacked, {len(all_images)-len(attack_list)} clean.")

    for img_name in all_images:
        img = cv2.imread(os.path.join(train_images_src, img_name))
        lbl_name = img_name.rsplit('.', 1)[0] + '.txt'

        shutil.copy(os.path.join(train_labels_src, lbl_name), os.path.join(robust_root, 'labels', lbl_name))

        # apply to only 50%
        if img_name in attack_list:
            with open(os.path.join(train_labels_src, lbl_name), 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 5:
                        bbox = [float(x) for x in parts[1:]]
                        img = apply_digital_tape(img, bbox)

        cv2.imwrite(os.path.join(robust_root, 'images', img_name), img)

create_robust_train_set('/content/datasets/archive', split_ratio=0.5, attack_type="tape")

Creating robust set: 657 attacked, 657 clean.


In [11]:
import yaml
# CREATE ATTACKED DATA YAML
robust_yaml_content = {
    'path': '/content/datasets/archive',
    'train': 'train_robust/images',
    'val': 'valid/images',
    'test': 'valid/images',
    'nc': 264,
    'names': ['0', '1', '100', '101', '102', '103', '105', '106', '107', '108', '109', '110', '111', '112', '113', '115', '116', '117', '12', '120', '121', '122', '123', '124', '125', '127', '128', '129', '131', '132', '133', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '153', '154', '155', '156', '158', '159', '16', '161', '163', '164', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '177', '178', '179', '18', '180', '181', '19', '2', '20', '21', '22', '23', '24', '27', '28', '29', '3', '31', '32', '33', '34', '35', '36', '39', '4', '40', '41', '42', '43', '45', '46', '47', '48', '49', '5', '50', '50 meters between vehicles', '51', '52', '53', '54', '55', '56', '57', '59', '60', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '73', '74', '76', '78', '79', '8', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '93', '94', '95', '97', '98', '99', 'Advance direction sign exit ahead from other road than motorway or expressway', 'Axle weight limit-2ton', 'Bus stop', 'Cattle', 'Crossroad intersection', 'Crossroads', 'Cycle track', 'Cyclist and mopeds rides on carrigeway', 'Cyclists', 'Dangerous shoulder', 'Dip', 'Direction sign exit sign', 'Direction to be followed', 'End of all restrictions', 'End of lane reserved for public transport', 'Falling rocks', 'Filling station', 'First aid post', 'Give way (Yield)', 'Give way -Yield-', 'Guarded level crossing ahead', 'Height limit-3.5m', 'Horn prohibited', 'Hotel', 'Keep left', 'Left curve', 'Level crossing countdown marker', 'Loose gravel', 'Main highways', 'Marking for sharp bends', 'Motorway', 'Narrow bridge', 'Narrow road', 'National border', 'No Left turn', 'No Right turn', 'No entry', 'No entry for Trucks', 'No entry for bicycles', 'No entry for bullock carts', 'No entry for hand carts', 'No entry for motor vehicles', 'No entry for pedestrians', 'No parking', 'No passing', 'No snowmobiles', 'No stopping', 'No vehicles exceeding 12 tonnes', 'No vehicles exceeding length shown', 'No vehicles in both directions', 'No vehicles or combination of vehicles exceeding weight shown', 'Oblique side road junction', 'One-way traffic', 'Parking', 'Parking allowed for 15min', 'Passing without stopping prohibited', 'Pedestrian crossing', 'Priority over oncoming vehicles', 'Refreshments', 'Restaurant', 'Restriction zone', 'Right curve', 'Road hump', 'Road number sign', 'Roadworks', 'Roundabout', 'School', 'Side road junction', 'Slippery road', 'Speed refulcation bump', 'Staggered side road junction', 'Steep ascent', 'Steep descent', 'Steep downhill', 'Steep uphill', 'Stop', 'Stop at customs', 'Straight ahead', 'Symbol plate for specified vehicle or road user category', 'T-junction', 'Telephone', 'Traffic signals', 'Turn Right', 'Turn left', 'Turn left ahead', 'Turn left or straight ahead', 'Turn right ahead', 'Uneven road', 'Unguarded level crossing ahead', 'Unprotected quay', 'Weight limit-5ton', 'Y-junction', 'adjoining way', 'axle weight limit 30tonnes', 'end of the speed limit', 'exit', 'falling rocks (from) left', 'falling rocks -from- left', 'length limit-10m', 'lowspeed zone', 'no entry for bicycles and humans', 'no motorcycles', 'no turning left', 'no u turn', 'snowmobiles', 'speed limit-100', 'speed limit-110', 'speed limit-30', 'speed limit-50', 'speed limit-60', 'speed limit-70', 'speed limit-80', 'speed limit-90', 'speed-limit-120', 'tunnel in 2 km', 'two way traffic', 'warning wild animal']
}

with open('/content/datasets/archive/robust.yaml', 'w') as f:
    yaml.dump(robust_yaml_content, f)

In [14]:
from ultralytics import YOLO

model = YOLO('/content/best.pt')

# Fine-tune on the 50/50 split
model.train(
    data='/content/datasets/archive/robust.yaml',
    epochs=50, # maybe try 20 epochs next time
    imgsz=640,
    batch=16,
    lr0=0.001,
    name='yolov8_robust_v1'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/archive/robust.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  1,   9,  20,  21,  26,  34,  64,  65,  72,  73,  87,  90, 102, 109, 111, 117, 118, 123, 140, 141, 152, 163, 164, 166, 168, 169, 176, 178, 183, 184, 200, 203, 212, 215, 217, 226, 230, 231, 233, 236, 237, 238, 257])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b95dea70830>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    

In [18]:
import yaml

attack_yaml_content = {
    'path': '/content/datasets/archive',
    'train': 'train_robust/images',
    'val': 'attacked_val/images',
    'test': 'attacked_val/images',
    'nc': 264,
    'names': ['0', '1', '100', '101', '102', '103', '105', '106', '107', '108', '109', '110', '111', '112', '113', '115', '116', '117', '12', '120', '121', '122', '123', '124', '125', '127', '128', '129', '131', '132', '133', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '153', '154', '155', '156', '158', '159', '16', '161', '163', '164', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '177', '178', '179', '18', '180', '181', '19', '2', '20', '21', '22', '23', '24', '27', '28', '29', '3', '31', '32', '33', '34', '35', '36', '39', '4', '40', '41', '42', '43', '45', '46', '47', '48', '49', '5', '50', '50 meters between vehicles', '51', '52', '53', '54', '55', '56', '57', '59', '60', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '73', '74', '76', '78', '79', '8', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '93', '94', '95', '97', '98', '99', 'Advance direction sign exit ahead from other road than motorway or expressway', 'Axle weight limit-2ton', 'Bus stop', 'Cattle', 'Crossroad intersection', 'Crossroads', 'Cycle track', 'Cyclist and mopeds rides on carrigeway', 'Cyclists', 'Dangerous shoulder', 'Dip', 'Direction sign exit sign', 'Direction to be followed', 'End of all restrictions', 'End of lane reserved for public transport', 'Falling rocks', 'Filling station', 'First aid post', 'Give way (Yield)', 'Give way -Yield-', 'Guarded level crossing ahead', 'Height limit-3.5m', 'Horn prohibited', 'Hotel', 'Keep left', 'Left curve', 'Level crossing countdown marker', 'Loose gravel', 'Main highways', 'Marking for sharp bends', 'Motorway', 'Narrow bridge', 'Narrow road', 'National border', 'No Left turn', 'No Right turn', 'No entry', 'No entry for Trucks', 'No entry for bicycles', 'No entry for bullock carts', 'No entry for hand carts', 'No entry for motor vehicles', 'No entry for pedestrians', 'No parking', 'No passing', 'No snowmobiles', 'No stopping', 'No vehicles exceeding 12 tonnes', 'No vehicles exceeding length shown', 'No vehicles in both directions', 'No vehicles or combination of vehicles exceeding weight shown', 'Oblique side road junction', 'One-way traffic', 'Parking', 'Parking allowed for 15min', 'Passing without stopping prohibited', 'Pedestrian crossing', 'Priority over oncoming vehicles', 'Refreshments', 'Restaurant', 'Restriction zone', 'Right curve', 'Road hump', 'Road number sign', 'Roadworks', 'Roundabout', 'School', 'Side road junction', 'Slippery road', 'Speed refulcation bump', 'Staggered side road junction', 'Steep ascent', 'Steep descent', 'Steep downhill', 'Steep uphill', 'Stop', 'Stop at customs', 'Straight ahead', 'Symbol plate for specified vehicle or road user category', 'T-junction', 'Telephone', 'Traffic signals', 'Turn Right', 'Turn left', 'Turn left ahead', 'Turn left or straight ahead', 'Turn right ahead', 'Uneven road', 'Unguarded level crossing ahead', 'Unprotected quay', 'Weight limit-5ton', 'Y-junction', 'adjoining way', 'axle weight limit 30tonnes', 'end of the speed limit', 'exit', 'falling rocks (from) left', 'falling rocks -from- left', 'length limit-10m', 'lowspeed zone', 'no entry for bicycles and humans', 'no motorcycles', 'no turning left', 'no u turn', 'snowmobiles', 'speed limit-100', 'speed limit-110', 'speed limit-30', 'speed limit-50', 'speed limit-60', 'speed limit-70', 'speed limit-80', 'speed limit-90', 'speed-limit-120', 'tunnel in 2 km', 'two way traffic', 'warning wild animal']

}

with open('/content/datasets/archive/attack.yaml', 'w') as f:
    yaml.dump(attack_yaml_content, f)

In [20]:
def create_attacked_val_set(base_path, attack_type="tape"):
    val_images_src = os.path.join(base_path, 'valid/images')
    val_labels_src = os.path.join(base_path, 'valid/labels')

    attacked_root = os.path.join(base_path, 'attacked_val')
    attacked_images_dest = os.path.join(attacked_root, 'images')
    attacked_labels_dest = os.path.join(attacked_root, 'labels')

    if os.path.exists(attacked_root):
        shutil.rmtree(attacked_root)

    os.makedirs(attacked_images_dest, exist_ok=True)
    os.makedirs(attacked_labels_dest, exist_ok=True)

    all_images = [f for f in os.listdir(val_images_src) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Attacking {len(all_images)} validation images")

    for img_name in all_images:
        img_path = os.path.join(val_images_src, img_name)
        img = cv2.imread(img_path)

        lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
        src_lbl_path = os.path.join(val_labels_src, lbl_name)
        dst_lbl_path = os.path.join(attacked_labels_dest, lbl_name)

        # 2. Use a safer copy check
        if os.path.exists(src_lbl_path):
            shutil.copy2(src_lbl_path, dst_lbl_path)

            # 3. Apply attack
            with open(src_lbl_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 5:
                        bbox = [float(x) for x in parts[1:]]
                        if attack_type == "tape":
                            img = apply_digital_tape(img, bbox)
                        elif attack_type == "patch":
                            img = apply_adversarial_patch(img, bbox)

        cv2.imwrite(os.path.join(attacked_images_dest, img_name), img)

    print(f"✅ Successfully created attacked validation set at locationnnn {attacked_root}")

create_attacked_val_set('/content/datasets/archive', attack_type="tape")

Attacking 39 validation images...
✅ Successfully created attacked validation set at /content/datasets/archive/attacked_val


In [21]:
robust_model = YOLO('/content/runs/detect/yolov8_robust_v1/weights/best.pt')

# validate against ATTACKED data
print("--ATTACKED DATA ---")
attack_results = robust_model.val(data='/content/datasets/archive/attack.yaml')

# validates against robust/50/50 data mix
print("---ROBUST DATA ---")
robust_results = robust_model.val(data='/content/datasets/archive/robust.yaml')

--- RESULTS ON ATTACKED DATA ---
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,390,416 parameters, 0 gradients, 9.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1661.3±500.9 MB/s, size: 73.5 KB)
val: Scanning /content/datasets/archive/attacked_val/labels... 39 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 39/39 2.3Kit/s 0.0s
val: New cache created: /content/datasets/archive/attacked_val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.8s
                   all         39         74      0.569      0.343      0.498      0.445
                     1          1          1        0.3          1      0.497      0.448
                   108          1          1          0          0      0.124     0.0871
                   121          1          2          0          0      0.153      0.145
                   122   